In [2]:
import json
import os
import pandas as pd
import glob
from transformers import AutoTokenizer
import re
import json
from dotenv import load_dotenv
from copy import deepcopy
from tqdm import tqdm
load_dotenv()
os.chdir(os.getenv('PARENT_DIR'))

/raid/home/m13521157/absa-sft-comparison/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
	"""
	Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
	Each dictionary contains the tag as the key and the corresponding value.
	For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
	[{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
	{'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

	Args:
		text (str): ABSA string output to be parsed.

	Returns:
		List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

	"""
	pattern = r"\[(\w+)\]\s*([^[]+)"
	matches = re.findall(pattern, text)

	result = []
	current_dict = {}

	for tag, content in matches:
		if tag == "SSEP":  # Sentence separator -> Start a new dictionary
			result.append(current_dict)
			current_dict = {}
		else:
			current_dict[tag] = content.strip()

	if current_dict:  # Append the last sentence if it exists
		result.append(current_dict)

	return result

In [42]:
dataset_type = f'hoasa'
dataset_folder = 'mvp_aos'
data_path = f'dataset/{dataset_type}/indo/{dataset_folder}/test.json'
with open(data_path, 'r') as f:
    data = json.load(f)
for i in range(len(data)):
	data[i]['dataset_type'] = dataset_type
data[0]

{'sentence_id': 2566,
 'instance_id': 284,
 'input': 'lumayan nyaman , tp kebersihan kmr mandi perlu ditingkatkan lg biar gk ada kuning2 di sudutnya lbh bgs [A] [O] [S]',
 'target': '[A] null [O] lumayan nyaman [S] positive [SSEP] [A] kebersihan kmr mandi [O] perlu ditingkatkan lg biar gk ada kuning2 di sudutnya lbh bgs [S] negative',
 'element_order': 'aos',
 'task_elements': 'aos',
 'dataset_type': 'hoasa'}

In [43]:
# Check for empty targets
mistakes = {
	"hoasa": [],
	"hotel_reviews": []
}
for instance in data:
	parsed_target = parse_absa_string(instance['target'])
	input_text = instance['input']
	if not parsed_target:
		print(f"No triplets found for Sentence ID {instance['sentence_id']}: {input_text}")
		mistakes[dataset_type].append(instance['sentence_id'])
mistakes

{'hoasa': [], 'hotel_reviews': []}

In [44]:
# Check for duplicated triplets
mistakes = {
	"hoasa": [],
	"hotel_reviews": []
}
for instance in data:
	parsed_target = parse_absa_string(instance['target'])
	input_text = instance['input']

	# Create a frequency dictionary to check for duplicates
	frequency = {}
	for d in parsed_target:
		key_tuple = tuple(d.items())
		if key_tuple in frequency:
			frequency[key_tuple] += 1
		else:
			frequency[key_tuple] = 1
	duplicated_entries = {k: v for k, v in frequency.items() if v > 1}

	if len(duplicated_entries) > 0:
		print(f"Duplicated triplets found for Sentence ID {instance['sentence_id']}: {input_text}")
		for entry, count in duplicated_entries.items():
			entry_dict = dict(entry)
			print(f"  Triplet: {entry_dict}, Count: {count}")
		mistakes[dataset_type].append(instance['sentence_id'])
mistakes


Duplicated triplets found for Sentence ID 2716: ini pertama kalinya saya memesan airy rooms . saya pesan tanggal 13april untuk menginap tanggal 28april . tp ketika saya sampai di hotel , kamar yg saya booking tidak ada dan pihak hotel mengembalikan uang saya , alhasil saya harus mencari hotel lain pada hari itu ! sangat mengecewakan karena tidak ada pemberitahuan dari jauh hari ! tidak profesional ! [A] [O] [S]
  Triplet: {'A': 'null', 'O': 'sangat mengecewakan', 'S': 'negative'}, Count: 2


{'hoasa': [2716], 'hotel_reviews': []}

In [ ]:
# First check
mistakes = {
	"hoasa": {
		'a': [],
		'o': [],
		's': [],
	},
	"hotel_reviews": {
		'a': [],
		'o': [],
		's': [],
	}
}
for instance in data:
	parsed_target = parse_absa_string(instance['target'])
	input_text = instance['input']

	# Check duplicates 

	for triplet in parsed_target:
		triplet_true = True
		aspect = triplet['A']
		if aspect not in input_text and aspect.lower() != 'null':
			print(f"Aspect Sentence ID {instance['sentence_id']}:\nAspect '{aspect}' not found in input: {input_text}")
			triplet_true = False
			mistakes[instance['dataset_type']]['a'].append(instance['sentence_id'])
		
		opinion = triplet['O']
		if opinion not in input_text and opinion.lower() != 'null':
			print(f"Opinion Sentence ID {instance['sentence_id']}:\nOpinion '{opinion}' not found in input: {input_text}")
			triplet_true = False
			mistakes[instance['dataset_type']]['o'].append(instance['sentence_id'])
		
		sentiment = triplet['S']
		if sentiment.lower() not in ['positive', 'negative']:
			print(f"Sentiment Sentence ID {instance['sentence_id']}:\nSentiment '{sentiment}' is not valid in input: {input_text}")
			triplet_true = False
			mistakes[instance['dataset_type']]['s'].append(instance['sentence_id'])
		
		if not triplet_true:
			print(f"Full Target: {instance['target']}")
			print(instance['dataset_type'])
			print("--------------------------------------------------")

In [174]:
for key, value in mistakes.items():
	print(f"Mistakes in {key}:")
	for tag, ids in value.items():
		unique_ids = set(ids)
		print(f" {len(unique_ids)} {tag.upper()} mistakes in sentence IDs: {sorted(unique_ids)}")

Mistakes in hoasa:
 0 A mistakes in sentence IDs: []
 0 O mistakes in sentence IDs: []
 0 S mistakes in sentence IDs: []
Mistakes in hotel_reviews:
 0 A mistakes in sentence IDs: []
 0 O mistakes in sentence IDs: []
 0 S mistakes in sentence IDs: []


In [175]:
# Second Check (word level match)
mistakes = {
	"hoasa": {
		'a': [],
		'o': [],
	},
	"hotel_reviews": {
		'a': [],
		'o': [],
	}
}
for instance in data:
	parsed_target = parse_absa_string(instance['target'])
	input_text = instance['input'].lower()
	input_text_words = input_text.split()

	for triplet in parsed_target:
		triplet_true = True
		aspect = triplet['A'].lower()
		if aspect != 'null':
			for word in aspect.split():
				if word not in input_text_words:
					print(f"Aspect Sentence ID {instance['sentence_id']}:\nAspect word '{word}' from aspect '{aspect}' not found in input: {input_text_words}")
					triplet_true = False
					mistakes[instance['dataset_type']]['a'].append(instance['sentence_id'])
		
		opinion = triplet['O'].lower()
		if opinion != 'null':
			for word in opinion.split():
				if word not in input_text_words:
					print(f"Opinion Sentence ID {instance['sentence_id']}:\nOpinion word '{word}' from opinion '{opinion}' not found in input: {input_text_words}")
					triplet_true = False
					mistakes[instance['dataset_type']]['o'].append(instance['sentence_id'])
		
		if not triplet_true:
			print(f"Full Target: {instance['target']}")
			print(instance['dataset_type'])
			print("--------------------------------------------------")

In [176]:
for key, value in mistakes.items():
	print(f"Probable mistakes in {key}:")
	for tag, ids in value.items():
		unique_ids = set(ids)
		print(f"  {tag.upper()} mistakes in sentence IDs: {len(unique_ids)} {sorted(unique_ids)}")

Probable mistakes in hoasa:
  A mistakes in sentence IDs: 0 []
  O mistakes in sentence IDs: 0 []
Probable mistakes in hotel_reviews:
  A mistakes in sentence IDs: 0 []
  O mistakes in sentence IDs: 0 []
